# CS 131 Project Assignment 3 - PySpark, Parquet, and DuckDB (Team 9)

Dataset: UCI Online Retail II (both workbook sheets combined into one TSV)

Team roles this sprint:
- Kanishka Yadav (Data Engineer)
- Sanjini Shukla (Data Engineer)
- Lipi Himanshukumar Makwana (Product Manager)
- Kushan Mehta (Data Storyteller)

Run from `pa3/notebooks/` with Java 17 or 21. Set `PA3_DATA_PATH` to the combined TSV if it is not in one of the
default locations checked in Section 0.

## 0. Environment setup and versions

Creates a local Spark session (`local[*]`, one machine) and prints the Spark, DuckDB, and Python versions.

- The session time zone is UTC, so timestamps are parsed the same way on every machine.
- ANSI mode is off, so an invalid value becomes `null` when cast and is counted in Section 3a instead of
  stopping the run. (Spark 4.x turns ANSI mode on by default.)

In [ ]:
# Import the Spark and Python libraries used in this notebook.
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from pyspark.sql import types as T
from pathlib import Path
import os
import sys
import platform
import pandas as pd
import duckdb

# Use the same Python executable for Spark workers and the notebook kernel.
os.environ.setdefault("PYSPARK_PYTHON", sys.executable)

# Create a local Spark session for this assignment.
spark = (
    SparkSession.builder
    .appName("CS131-PA3-Online-Retail")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.ansi.enabled", "false")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print("Spark version:", spark.version)
print("DuckDB version:", duckdb.__version__)
print("Python version:", platform.python_version())
print("Spark master:", spark.sparkContext.master)
print("Spark ran locally:", spark.sparkContext.master.startswith("local"))


### 0a. Paths

Run from `pa3/notebooks/`. The input is the first one found:

1. `PA3_DATA_PATH`, if set (use this for the full TSV in a custom location).
2. The full combined TSV in a default location outside the repo.
3. The committed 1,000-row sample, `pa3/data/sample/online_retail_sample.tsv`, for development or for running the
   notebook without the full dataset.

A sample run writes to `pa3/out/sample_run/` and `pa3/data/parquet/sample_run/` (both gitignored), so it never
overwrites the full-data outputs in `pa3/out/`.

On IBM:

```bash
export PA3_DATA_PATH=../online_retail_II_combined.tsv
```

In [ ]:
# Define paths relative to the pa3 directory.
PA3_DIR = Path.cwd().parent
REPO_DIR = PA3_DIR.parent

SAMPLE_PATH = PA3_DIR / "data" / "sample" / "online_retail_sample.tsv"
FULL_ROWS = 1_067_371

# Use PA3_DATA_PATH when the full dataset is stored in a custom location.
DATA_PATH = os.environ.get("PA3_DATA_PATH")

if DATA_PATH:
    DATA_PATH = Path(DATA_PATH)
else:
    candidates = [
        REPO_DIR.parent / "online_retail_II_combined.tsv",
        REPO_DIR.parent / "data" / "online_retail_II_combined.tsv",
        SAMPLE_PATH,
    ]
    DATA_PATH = next((p for p in candidates if p.exists()), None)

if DATA_PATH is None or not DATA_PATH.exists():
    raise FileNotFoundError(
        "Set PA3_DATA_PATH to the combined Online Retail II TSV file, "
        f"or run from pa3/notebooks/ so the committed sample ({SAMPLE_PATH}) is found."
    )

USING_SAMPLE = DATA_PATH.resolve() == SAMPLE_PATH.resolve()

OUT_DIR = PA3_DIR / "out"
PARQUET_DIR = PA3_DIR / "data" / "parquet" / "online_retail_prepared"
DEV_SAMPLE_DIR = PA3_DIR / "data" / "dev_sample_20pct"

# Keep sample-run files separate from the full-data deliverables.
if USING_SAMPLE:
    OUT_DIR = PA3_DIR / "out" / "sample_run"
    PARQUET_DIR = PA3_DIR / "data" / "parquet" / "sample_run" / "online_retail_prepared"
    DEV_SAMPLE_DIR = PA3_DIR / "data" / "parquet" / "sample_run" / "dev_sample_20pct"

OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Input:", DATA_PATH)
print("Input size (bytes):", f"{DATA_PATH.stat().st_size:,}")
print("Using committed 1,000-row sample:", USING_SAMPLE)
print("Outputs:", OUT_DIR)


## 1. Project goal, questions, and data description

**Lipi:** To be completed. Include the project goal, the two questions, row meaning, key, important columns,
data types, and units. The data types must match the schema contract in Section 3b.

## 2. Read the data with PySpark

Reads the combined Online Retail II TSV (1,067,371 transaction rows plus a header) and shows its schema.

- **Explicit string schema:** all eight columns are read as strings. Section 3 casts them, so cast failures can be counted.
- **`quote=""`:** quoting is off because some descriptions contain literal `"` characters. Each line is split on
  tabs, as in PA2.
- **`mode="FAILFAST"`:** the read stops if any line does not have eight fields.
- Empty fields are read as `null`.

In [ ]:
# Read the TSV with an explicit string schema.
RAW_COLUMNS = [
    "Invoice",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "Price",
    "Customer ID",
    "Country",
]

raw_schema = T.StructType([
    T.StructField(column, T.StringType(), True)
    for column in RAW_COLUMNS
])

raw_df = (
    spark.read
    .option("header", True)
    .option("sep", "\t")
    .option("quote", "")
    .option("mode", "FAILFAST")
    .schema(raw_schema)
    .csv(str(DATA_PATH))
)

raw_rows = raw_df.count()
print("Rows:", raw_rows)

if USING_SAMPLE:
    assert raw_rows == 1_000, f"Expected 1,000 sample rows, got {raw_rows:,}"
else:
    assert raw_rows == FULL_ROWS, f"Expected {FULL_ROWS:,} rows, got {raw_rows:,}"

raw_df.printSchema()
raw_df.show(5, truncate=False)


## 3. Select and cast columns

Renames the columns to snake_case and casts them to analysis types. Text and ID columns are not trimmed or
re-cased: `stock_code` stays case-sensitive (`85123A` and `85123a` are different codes) and untrimmed, matching
how PA2 counted stock codes.

| Column | Source | Type | Unit / meaning |
|---|---|---|---|
| `invoice` | Invoice | string | Invoice number; a leading `C` marks a cancellation |
| `stock_code` | StockCode | string (unchanged) | Product code |
| `description` | Description | string | Product name |
| `quantity` | Quantity | integer | Units of the product on this invoice line |
| `invoice_date` | InvoiceDate | timestamp (`yyyy-MM-dd HH:mm:ss`) | Invoice date and time |
| `price` | Price | double | Price per unit, pounds sterling (GBP) |
| `customer_id` | Customer ID | string | Customer identifier (an ID, not a measure) |
| `country` | Country | string (unchanged) | Customer's country |

**Row meaning:** one row is one invoice line (one product on one invoice).

In [ ]:
# Select the needed columns and cast them to analysis types.
typed_df = raw_df.select(
    F.col("Invoice").alias("invoice"),
    F.col("StockCode").alias("stock_code"),
    F.col("Description").alias("description"),
    F.col("Quantity").cast("int").alias("quantity"),
    F.to_timestamp(
        F.col("InvoiceDate"),
        "yyyy-MM-dd HH:mm:ss"
    ).alias("invoice_date"),
    F.col("Price").cast("double").alias("price"),
    F.col("Customer ID").alias("customer_id"),
    F.col("Country").alias("country"),
)

typed_df.printSchema()
typed_df.show(5, truncate=False)


### 3a. Cast failures and missing key values

A **cast failure** is a value that is present in the raw data but `null` after casting. A **blank** is a string
that is empty after trimming. Both are counted; no values are changed. PySpark and DuckDB apply the same rules
because both read the same Parquet.

| Case | Rule |
|---|---|
| Null or blank `stock_code` or `country` | Expected 0 (asserted). If any appeared, Analysis 1 would keep them as their own group, as PA2's `uniq -c` did. `Unspecified` is a real country value. |
| Invalid `quantity`, `price`, or `invoice_date` | Expected 0 cast failures (asserted). |
| Missing `customer_id` | Kept and flagged in Section 5. |

In [ ]:
# Count values that fail during casting or are missing from the raw input.
checks = raw_df.agg(
    F.sum(F.when(F.col("Quantity").isNotNull() & F.col("Quantity").cast("int").isNull(), 1).otherwise(0)).alias("quantity_cast_failures"),
    F.sum(F.when(F.col("Price").isNotNull() & F.col("Price").cast("double").isNull(), 1).otherwise(0)).alias("price_cast_failures"),
    F.sum(F.when(F.col("InvoiceDate").isNotNull() & F.to_timestamp(F.col("InvoiceDate"), "yyyy-MM-dd HH:mm:ss").isNull(), 1).otherwise(0)).alias("invoice_date_cast_failures"),
    F.sum(F.when(F.col("Quantity").isNull(), 1).otherwise(0)).alias("null_quantity"),
    F.sum(F.when(F.col("Price").isNull(), 1).otherwise(0)).alias("null_price"),
    F.sum(F.when(F.col("InvoiceDate").isNull(), 1).otherwise(0)).alias("null_invoice_date"),
    F.sum(F.when(F.col("StockCode").isNull(), 1).otherwise(0)).alias("null_stock_code"),
    F.sum(F.when(F.trim(F.col("StockCode")) == "", 1).otherwise(0)).alias("blank_stock_code"),
    F.sum(F.when(F.col("Country").isNull(), 1).otherwise(0)).alias("null_country"),
    F.sum(F.when(F.trim(F.col("Country")) == "", 1).otherwise(0)).alias("blank_country"),
    F.sum(F.when(F.col("Customer ID").isNull() | (F.trim(F.col("Customer ID")) == ""), 1).otherwise(0)).alias("missing_customer_id"),
    F.sum(F.when(F.col("StockCode") != F.trim(F.col("StockCode")), 1).otherwise(0)).alias("stock_code_with_edge_whitespace"),
).first().asDict()

for name, value in checks.items():
    print(f"{name:34s} {value:>10,}")

must_be_zero = [
    "quantity_cast_failures", "price_cast_failures", "invoice_date_cast_failures",
    "null_quantity", "null_price", "null_invoice_date",
    "null_stock_code", "blank_stock_code", "null_country", "blank_country",
]
nonzero = {name: checks[name] for name in must_be_zero if checks[name] != 0}
assert not nonzero, f"Unexpected missing or invalid values: {nonzero}"
print("\nAll cast-failure and missing-key checks are zero.")


### 3b. Schema contract for downstream sections

Sections 4–8 add columns to this schema. **Sections 9–12 must use `prepared`, the DataFrame reloaded from Parquet
in Section 7**, not `typed_df`.

| Column | Type | Added in | Notes |
|---|---|---|---|
| `invoice` | string | 3 | Unchanged |
| `stock_code` | string | 3 | **Unchanged — do not trim or change case** |
| `description` | string | 3 | Literal text (quoting was off, so any `"` characters are kept) |
| `quantity` | int | 3 | Units |
| `invoice_date` | timestamp | 3 | Session time zone is UTC |
| `price` | double | 3 | GBP per unit |
| `customer_id` | string | 3 | `null` when missing |
| `country` | string | 3 | Unchanged |
| `revenue` | double | 4 | `quantity * price`, GBP; `null` if either input is `null`; negative for returns |
| `is_cancellation` | boolean | 4 | `invoice` starts with `C` |
| `flag_missing_customer` | boolean | 5 | `customer_id` is null or blank |
| `flag_bad_qty_price` | boolean | 5 | `quantity` or `price` is `null` or ≤ 0 |

Parquet location (generated locally, gitignored): `pa3/data/parquet/online_retail_prepared/`.

## 4. Derived columns

Adds two columns used by the analyses. Nothing is rounded here; rounding happens only in exported results.

| Column | Type | Rule |
|---|---|---|
| `revenue` | double | `quantity * price` in GBP; `null` if either input is `null`; negative for returns |
| `is_cancellation` | boolean | `invoice` starts with `C` |

In [ ]:
# Add the derived revenue and cancellation columns.
derived_df = (
    typed_df
    .withColumn("revenue", F.col("quantity") * F.col("price"))
    .withColumn("is_cancellation", F.col("invoice").startswith("C"))
)

derived_df.select("invoice", "stock_code", "quantity", "price", "revenue", "is_cancellation") \
    .orderBy(F.col("is_cancellation").desc(), "invoice") \
    .show(5, truncate=False)

# Check the derived revenue value on a few rows.
for row in derived_df.select("quantity", "price", "revenue").limit(5).collect():
    assert abs(row["quantity"] * row["price"] - row["revenue"]) < 1e-9
print("revenue spot check passed")

## 5. Data-quality checks

Adds two quality flags and counts the affected rows. No rows are removed.

**Key:** the data has no natural key. There is no line-item ID, and `(invoice, stock_code)` repeats on some
invoices (counted below).

| Check | Rule | Decision |
|---|---|---|
| A. Missing customer | `customer_id` is null or blank | Kept and flagged (`flag_missing_customer`). These are real sales without a customer account; customer-level work must filter them out. |
| B. Bad quantity or price | `quantity` or `price` is null or ≤ 0 | Kept and flagged (`flag_bad_qty_price`). Mostly cancellations and returns. PA1's stock-code counts included them, so Analysis 1 keeps them. Use `NOT flag_bad_qty_price` for completed purchases only (PA2's filter). |

**Overlap:** rows with at least one problem = A + B − (A and B), not A + B. Both the overlap and the union are reported.

**Exact duplicates are kept.** Without a line-item ID, identical rows may be the same product scanned twice on
one invoice. They are counted as duplicate groups (PA2's 32,907 figure) and as extra copies.

In [ ]:
# Add quality flags without removing rows.
flagged_df = (
    derived_df
    .withColumn(
        "flag_missing_customer",
        F.col("customer_id").isNull()
        | (F.trim(F.col("customer_id")) == ""),
    )
    .withColumn(
        "flag_bad_qty_price",
        F.col("quantity").isNull() | F.col("price").isNull()
        | (F.col("quantity") <= 0) | (F.col("price") <= 0),
    )
)

total_rows = flagged_df.count()

flag_counts = flagged_df.agg(
    F.sum(F.col("flag_missing_customer").cast("int")).alias("a"),
    F.sum(F.col("flag_bad_qty_price").cast("int")).alias("b"),
    F.sum((F.col("flag_missing_customer") & F.col("flag_bad_qty_price")).cast("int")).alias("a_and_b"),
    F.sum((F.col("flag_missing_customer") | F.col("flag_bad_qty_price")).cast("int")).alias("a_or_b"),
    F.sum(F.col("is_cancellation").cast("int")).alias("cancel"),
    F.sum((F.col("is_cancellation") & F.col("flag_bad_qty_price")).cast("int")).alias("cancel_and_b"),
).first()

# Subtract the overlap once when calculating the union.
assert flag_counts["a_or_b"] == flag_counts["a"] + flag_counts["b"] - flag_counts["a_and_b"]

# Count repeated complete rows without deleting them.
dup_groups = (
    typed_df.groupBy(typed_df.columns).count()
    .filter(F.col("count") > 1)
    .agg(F.count("*").alias("groups"), F.sum(F.col("count") - 1).alias("extra_copies"))
    .first()
)
dup_group_count = dup_groups["groups"] or 0
dup_extra_copies = dup_groups["extra_copies"] or 0
assert total_rows - typed_df.dropDuplicates().count() == dup_extra_copies

# Check whether invoice and stock_code form a unique key.
pair_rows = (
    typed_df.groupBy("invoice", "stock_code").count()
    .filter(F.col("count") > 1)
    .agg(F.sum("count").alias("rows"))
    .first()["rows"] or 0
)

for name, value in [
    ("total rows", total_rows),
    ("A. missing customer_id", flag_counts["a"]),
    ("B. quantity/price null or <= 0", flag_counts["b"]),
    ("A and B (overlap)", flag_counts["a_and_b"]),
    ("A or B (union)", flag_counts["a_or_b"]),
    ("cancellations (invoice starts with C)", flag_counts["cancel"]),
    ("cancellations that also fail B", flag_counts["cancel_and_b"]),
    ("exact duplicate groups", dup_group_count),
    ("exact duplicate extra copies", dup_extra_copies),
    ("rows sharing (invoice, stock_code)", pair_rows),
]:
    print(f"{name:40s} {value:>10,}")

### 5a. Save `out/quality_checks.csv`

One row per check, with its share of all rows. `overlap_note` shows where checks overlap, so the counts are not
simply added.

In [ ]:
def pct(n):
    return round(100.0 * n / total_rows, 2) if total_rows else 0.0

quality_rows = [
    ("Q0", "total_rows", "All rows read from the input", total_rows,
     "Population for every percentage below", ""),
    ("Q1", "cast_failures", "Raw quantity, price, or invoice_date present but null after casting",
     checks["quantity_cast_failures"] + checks["price_cast_failures"] + checks["invoice_date_cast_failures"],
     "Asserted to be 0 (Section 3a)", ""),
    ("Q2", "missing_customer_id", "customer_id is null or blank", flag_counts["a"],
     "Kept; flag_missing_customer = true", f"{flag_counts['a_and_b']:,} of these rows also fail Q3"),
    ("Q3", "nonpositive_or_missing_qty_price", "quantity or price is null or <= 0", flag_counts["b"],
     "Kept; flag_bad_qty_price = true", f"{flag_counts['a_and_b']:,} of these rows also fail Q2"),
    ("Q4", "overlap_q2_and_q3", "Rows failing both Q2 and Q3", flag_counts["a_and_b"],
     "Informational", "Counted in both Q2 and Q3"),
    ("Q5", "union_q2_or_q3", "Rows failing Q2 or Q3 (at least one)", flag_counts["a_or_b"],
     "Informational", "Q2 + Q3 - Q4; not Q2 + Q3"),
    ("Q6", "cancellation_rows", "invoice starts with 'C'", flag_counts["cancel"],
     "Kept; is_cancellation = true", f"{flag_counts['cancel_and_b']:,} of these rows also fail Q3"),
    ("Q7", "exact_duplicate_extra_copies", "Rows identical in all 8 columns to an earlier row", dup_extra_copies,
     "Kept; cannot prove true duplicates without a line-item ID",
     f"Spread over {dup_group_count:,} duplicate groups (PA2 reported the group count)"),
    ("Q8", "rows_sharing_invoice_stock_code", "(invoice, stock_code) appears on more than one row", pair_rows,
     "Informational; shows (invoice, stock_code) is not a key", "Includes all Q7 rows"),
    ("Q9", "stock_code_edge_whitespace", "stock_code has leading or trailing whitespace",
     checks["stock_code_with_edge_whitespace"], "Kept unchanged to match PA2 grouping", ""),
]

quality_pdf = pd.DataFrame(
    quality_rows,
    columns=["check_id", "check_name", "rule", "rows_affected", "action", "overlap_note"],
)
quality_pdf.insert(4, "pct_of_rows", quality_pdf["rows_affected"].map(pct))

quality_pdf.to_csv(OUT_DIR / "quality_checks.csv", index=False)
print("Wrote", OUT_DIR / "quality_checks.csv")
quality_pdf

## 6. Write Parquet

Writes the prepared data (8 cast columns, 2 derived columns, 2 flags) to Parquet, replacing any earlier run. The
Parquet folder is gitignored.

In [ ]:
# Select the prepared columns and write them as Parquet.
prepared_columns = [
    "invoice", "stock_code", "description", "quantity", "invoice_date", "price",
    "customer_id", "country", "revenue", "is_cancellation",
    "flag_missing_customer", "flag_bad_qty_price",
]
prepared_to_write = flagged_df.select(*prepared_columns)

prepared_to_write.write.mode("overwrite").parquet(str(PARQUET_DIR))

parquet_bytes = sum(p.stat().st_size for p in PARQUET_DIR.rglob("*.parquet"))
print("Wrote", PARQUET_DIR)
print("Parquet size (bytes):", f"{parquet_bytes:,}")

## 7. Read Parquet back and verify

Reads the Parquet back as `prepared` and checks that the row count, column order, and data types are unchanged.
Sections 8–12 use `prepared`.

In [ ]:
# Read the prepared Parquet data back into a new DataFrame.
prepared = spark.read.parquet(str(PARQUET_DIR))

prepared_rows = prepared.count()
expected_types = [(f.name, f.dataType.simpleString()) for f in prepared_to_write.schema.fields]
reloaded_types = [(f.name, f.dataType.simpleString()) for f in prepared.schema.fields]

print("Rows before write:", f"{total_rows:,}")
print("Rows after reload:", f"{prepared_rows:,}")
prepared.printSchema()

assert prepared_rows == total_rows, "Row count changed after the Parquet round trip"
assert reloaded_types == expected_types, f"Schema changed:\n{expected_types}\n{reloaded_types}"
assert prepared.filter(F.col("flag_missing_customer")).count() == flag_counts["a"]
print("Parquet round trip verified: same row count, column order, and data types.")

## 8. 20% development sample and sample comparison

Creates the development sample from `prepared` with the class method:

```python
sample = prepared.sample(withReplacement=False, fraction=0.20, seed=131)
```

- The sample has about 20% of the rows, not exactly 20%. The same input, seed, and machine give the same sample.
- It is for development only. Analyses 1–2 and the benchmark use the full `prepared` data.
- The comparison covers `country` (every country, so missing groups are listed), `is_cancellation`, and
  `flag_missing_customer`. `pct_point_diff` is sample % minus population %.

In [ ]:
# Create the development sample using the class sampling pattern.
SAMPLE_FRACTION = 0.20
SAMPLE_SEED = 131

sample = prepared.sample(withReplacement=False, fraction=SAMPLE_FRACTION, seed=SAMPLE_SEED)

population_rows = prepared_rows
sample_rows = sample.count()
assert sample_rows > 0, "The development sample is empty"

print("Fraction:", SAMPLE_FRACTION, "| seed:", SAMPLE_SEED)
print("Population rows:", f"{population_rows:,}")
print("Sample rows:    ", f"{sample_rows:,}", f"({100 * sample_rows / population_rows:.2f}% of population)")

sample.write.mode("overwrite").parquet(str(DEV_SAMPLE_DIR))
print("Saved development sample to", DEV_SAMPLE_DIR)

In [ ]:
# Compare population and sample proportions for useful groups.
def compare_group(column):
    population = prepared.groupBy(column).agg(F.count("*").alias("population_rows"))
    sample_counts = sample.groupBy(column).agg(F.count("*").alias("sample_rows"))
    return (
        population.join(sample_counts, on=column, how="left")
        .fillna(0, subset=["sample_rows"])
        .select(
            F.lit(column).alias("group_column"),
            F.col(column).cast("string").alias("group_value"),
            "population_rows",
            "sample_rows",
        )
    )


comparison_pdf = pd.concat(
    [compare_group(c).toPandas() for c in ["country", "is_cancellation", "flag_missing_customer"]],
    ignore_index=True,
)
comparison_pdf["population_pct"] = (100 * comparison_pdf["population_rows"] / population_rows).round(4)
comparison_pdf["sample_pct"] = (100 * comparison_pdf["sample_rows"] / sample_rows).round(4)
comparison_pdf["pct_point_diff"] = (comparison_pdf["sample_pct"] - comparison_pdf["population_pct"]).round(4)
comparison_pdf["in_sample"] = comparison_pdf["sample_rows"] > 0
comparison_pdf.insert(0, "seed", SAMPLE_SEED)
comparison_pdf.insert(0, "fraction", SAMPLE_FRACTION)

# Deterministic order: grouping, then largest population group first, then name.
comparison_pdf = comparison_pdf.sort_values(
    ["group_column", "population_rows", "group_value"], ascending=[True, False, True]
).reset_index(drop=True)

# Check that each grouping accounts for every population and sample row.
for column, part in comparison_pdf.groupby("group_column"):
    assert part["population_rows"].sum() == population_rows, column
    assert part["sample_rows"].sum() == sample_rows, column

comparison_pdf.to_csv(OUT_DIR / "sample_comparison.csv", index=False)
print("Wrote", OUT_DIR / "sample_comparison.csv")

missing = comparison_pdf.loc[~comparison_pdf["in_sample"], ["group_column", "group_value", "population_rows"]]
if missing.empty:
    print("Every group in the population also appears in the sample.")
else:
    print("Groups present in the population but missing from the sample:")
    print(missing.to_string(index=False))

print("Largest absolute difference (percentage points):", comparison_pdf["pct_point_diff"].abs().max())
comparison_pdf[comparison_pdf["group_column"] != "country"]

## Handoff to Sanjini - Sections 9-12 and final integration

Sections 0-8 are finished and verified on the full dataset. **From this point on, only Sanjini edits this
notebook.** Lipi and Kushan send their text and chart code to Sanjini, who pastes them in.

**Ready to use (defined in Sections 0-8):**

| Name | What it is |
|---|---|
| `prepared` | Full prepared data reloaded from Parquet (1,067,371 rows). **Use this for Sections 9-12, not `typed_df` or the sample.** |
| `PARQUET_DIR` | Folder of the prepared Parquet files (DuckDB reads `PARQUET_DIR / "*.parquet"`) |
| `parquet_bytes` | Total size of the Parquet part files; this is `input_bytes` in the benchmark |
| `prepared_rows` | Row count of `prepared`; this is `input_rows` in the benchmark |
| `OUT_DIR` | Output folder (`pa3/out/`, or `pa3/out/sample_run/` on a sample run) |

**Rules for every section below:**
- Never trim, upper-case, or lower-case `stock_code` or `country`.
- Export results with `df.toPandas().to_csv(OUT_DIR / "name.csv", index=False)`. This writes one real CSV with a
  header. Do not use `df.write.csv(...)`, which writes a folder of part files.
- The required file names are exactly `analysis_1.csv`, `analysis_2.csv`, and `benchmark.csv` (only these are
  unignored in `.gitignore`).

**Final integration checklist (Saturday):**
- [ ] Paste in Lipi's Section 1 and Section 15 text and Kushan's Section 13-14 content.
- [ ] Kernel -> Restart & Run All on the **full** TSV (Section 0a must print
  `Using committed 1,000-row sample: False`).
- [ ] All five CSVs in `pa3/out/` are regenerated by that run, and no cell shows an error.

## 9. Analysis 1 - stock-code frequency (PySpark)

**Sanjini:** To be completed.

**Goal:** reproduce PA2's stock-code frequency table (`cut -f2 | sort | uniq -c | sort -nr`).

**What to build:**
- Use **all rows** of `prepared` (PA2 counted every row, including cancellations and missing customers).
- `groupBy("stock_code")` and count the rows as `transaction_rows`.
- Order by `transaction_rows` descending, then `stock_code` ascending (tie rule).
- Export every row to `analysis_1.csv` with columns `stock_code, transaction_rows`.

**Acceptance checks (add `assert`s):**
- The result has **5,305 rows** (distinct stock codes in the full data).
- All 20 rows below match on both stock code and count. PA2's Top 20 has no ties (the first tie is at rank 56),
  so the order must match too.

| Rank | stock_code | transaction_rows | Rank | stock_code | transaction_rows |
|---:|---|---:|---:|---|---:|
| 1 | 85123A | 5829 | 11 | 20727 | 2529 |
| 2 | 22423 | 4424 | 12 | 21931 | 2434 |
| 3 | 85099B | 4216 | 13 | 22386 | 2347 |
| 4 | 21212 | 3318 | 14 | 22469 | 2325 |
| 5 | 20725 | 3259 | 15 | 22411 | 2297 |
| 6 | 84879 | 2960 | 16 | 84991 | 2271 |
| 7 | 47566 | 2768 | 17 | 22382 | 2251 |
| 8 | 21232 | 2747 | 18 | 22384 | 2230 |
| 9 | 22197 | 2549 | 19 | 21080 | 2224 |
| 10 | 22383 | 2540 | 20 | 22086 | 2217 |

Source: the PA2 pipeline `tail -n +2 INPUT | cut -f2 | sort | uniq -c | sort -nr | head -n 20` on the full
combined TSV. Rows 1-10 match the PA2 report table. Store this list in the notebook as a Python list and compare
it with the first 20 rows of the result.

## 10. Analysis 2 - top three stock codes per country (window)

**Sanjini:** To be completed.

**Goal:** the three most frequent stock codes in each country, by row count.

**What to build** (class window pattern):

```python
counts = (
    prepared
    .filter(
        F.col("country").isNotNull() &
        F.col("stock_code").isNotNull()
    )
    .groupBy("country", "stock_code")
    .agg(F.count("*").alias("transaction_rows"))
)

country_window = Window.partitionBy("country").orderBy(
    F.desc("transaction_rows"),
    F.asc("stock_code")
)

analysis_2 = (
    counts
    .withColumn("rank", F.row_number().over(country_window))
    .filter(F.col("rank") <= 3)
    .select("country", "rank", "stock_code", "transaction_rows")
    .orderBy("country", "rank")
)
```

- Same rows as Analysis 1. The null filter is the documented rule (null `country` or `stock_code` rows are
  excluded here); there are 0 such rows, as Section 3a asserts.
- `Unspecified` is a real country value and keeps its own top 3.
- Export to `analysis_2.csv` with columns `country, rank, stock_code, transaction_rows`.

**Explain in a markdown cell:** `partitionBy("country")` ranks each country separately; `orderBy` puts the
most frequent codes first; `row_number()` gives exactly one rank per row, so a tie is broken by `stock_code`
ascending and each country gets at most 3 rows. A window is needed because `groupBy` alone cannot rank rows
inside each group while keeping the row-level detail.

**Acceptance checks:**
- No country has more than 3 rows; ranks are 1, 2, 3 with no gaps.
- Check one small country by hand on the TSV and record the command and result in the notebook. Bermuda is a
  good choice: every one of its stock codes appears once, so its top 3 also tests the tie rule
  (expected `16156S`, `20702`, `20703`). For example:

```bash
awk -F'\t' 'NR>1 && $8=="Bermuda" {print $2}' ../online_retail_II_combined.tsv | LC_ALL=C sort | uniq -c | LC_ALL=C sort -k1,1nr -k2,2 | head -3
```

## 11. Analysis 1 in DuckDB and row-by-row comparison

**Sanjini:** To be completed.

**What to build:**
- Run the same query in DuckDB on the **same Parquet** that Spark reads:

```sql
SELECT stock_code, COUNT(*) AS transaction_rows
FROM read_parquet('<PARQUET_DIR>/*.parquet')
GROUP BY stock_code
ORDER BY transaction_rows DESC, stock_code ASC
```

- Load both results into pandas and compare **every row by joining on `stock_code`** (an outer merge), not by
  position, so a different tie order can never cause a false mismatch.
- Tolerance is **0**: the counts are integers, so they must match exactly. State this in a markdown cell.

**Print and assert:** both row counts (5,305), stock codes only in Spark (0), stock codes only in DuckDB (0),
and rows where the counts differ (0). Save the overall pass/fail as a variable (for example
`results_match`); Section 12 uses it for `result_verified`.

## 12. Benchmark

**Sanjini:** To be completed.

**`benchmark.csv` must have exactly this header and 2 rows** (one PySpark, one DuckDB):

```text
implementation,environment,input_rows,input_bytes,format,elapsed_seconds,output_rows,result_verified
```

| Column | Value |
|---|---|
| `implementation` | `pyspark` or `duckdb` |
| `environment` | Built from code, e.g. `Spark 3.5.3 local[*], macOS, 10 cores` / `DuckDB 1.1.3, same machine` |
| `input_rows` | `prepared_rows` |
| `input_bytes` | `parquet_bytes` (both engines read the same Parquet) |
| `format` | `parquet` |
| `elapsed_seconds` | **One timed run each** with `time.perf_counter()` |
| `output_rows` | Number of result rows returned (5,305 expected) |
| `result_verified` | `True` only if the Section 11 comparison passed |

**What the timer covers (the same for both engines):** reading the Parquet, the Analysis 1 aggregation, and
returning the result to Python: `.collect()` for Spark (Spark is lazy, so without an action nothing runs) and
`.fetchall()` for DuckDB. It does **not** include starting the SparkSession or writing CSVs. Do not `.cache()`
anything before timing. No repeated runs and no averages.

**Write in a markdown cell:** the machine and local mode, that both engines read the same Parquet, which engine
ran first, and that a single run can favor the second engine because the files may already be in the operating
system's file cache.

## 13. Chart

**Kushan:** To be completed.

## 14. Findings and limitations

**Kushan:** To be completed.

## 15. AI-use disclosure

**Lipi:** To be completed. Collect the AI-use disclosure from all members.